# 1. What is Data Leakage?

### Concept & Definition
Data leakage occurs when information from outside the training dataset (such as target labels or future evaluation metrics) contaminates the training phase.

### Primary Causes:
1. **Target Leakage:** Including features created after the outcome occurs.
2. **Train-Test Contamination:** Fitting scalers or imputers on the full dataset before splitting.
3. **Temporal Leakage:** Using future observations to predict past outcomes.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Cleaned_Validated_Data.csv")

# 2. Incorrect vs. Correct Preprocessing Workflow

### Incorrect (Leaky) Workflow:
1. Fit `StandardScaler` / `SimpleImputer` on the entire dataset ($X$).
2. Perform train/test split.
*Result:* Test set statistics leak into the training scaler calculations.

### Correct Workflow:
1. Perform train/test split FIRST.
2. Fit transformers ONLY on $X_{train}$.
3. Transform $X_{train}$ and $X_{test}$ separately using learned parameters.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["MonthlyCharges"]].fillna(df["MonthlyCharges"].median())
y = df["Churn"]

# ==================== INCORRECT WORKFLOW ====================
scaler_leaky = StandardScaler()
X_scaled_leaky = scaler_leaky.fit_transform(X) # LEAK! Fit on full dataset
X_tr_bad, X_ts_bad, y_tr_bad, y_ts_bad = train_test_split(X_scaled_leaky, y, test_size=0.2, random_state=42)

# ==================== CORRECT WORKFLOW ====================
X_tr_good, X_ts_good, y_tr_good, y_ts_good = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_correct = StandardScaler()

X_tr_good_scaled = scaler_correct.fit_transform(X_tr_good) # FIT ONLY ON TRAIN
X_ts_good_scaled = scaler_correct.transform(X_ts_good)    # TRANSFORM TEST

print("Leaky Mean (Full Dataset):", scaler_leaky.mean_[0].round(4))
print("Correct Mean (Train Split Only):", scaler_correct.mean_[0].round(4))
print("Notebook 13 execution completed successfully!")

Leaky Mean (Full Dataset): 64.1021
Correct Mean (Train Split Only): 63.493
Notebook 13 execution completed successfully!
